# AdaBoost for classification

[Master Tree Ensemble Models course](https://www.trainindata.com/p/master-tree-ensemble-models)

In this notebook, we train an AdaBoost to predict which clients subscribe to a term deposit. We look at the predictions that the model returns, then we see how the performance changes with the number of trees, and finally we explore the trade off between the learning rate and the size of the ensemble.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

### Load data

We use the bank marketing data that we prepared in the notebook of the folder prepare-data.

In [ ]:
df = pd.read_csv("../prepare-data/bank_marketing.csv")
df.head()

In [ ]:
# the trees from Scikit-learn take numbers, so we replace the categories by integers
categorical_vars = df.select_dtypes("str").columns
df[categorical_vars] = df[categorical_vars].apply(lambda var: var.astype("category").cat.codes)

df.head()

In [ ]:
# AdaBoost does not take missing values, so we replace them by a value outside the distribution
df["pdays"] = df["pdays"].fillna(-1)

df.isnull().sum().sum()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns="y"),
    df["y"],
    test_size=0.3,
    stratify=df["y"],
    random_state=0,
)

X_train.shape, X_test.shape

### Train the model with the default parameters

We show the hyperparameters with their default values, so that we can see the effect of changing them later on. When we leave the estimator empty, AdaBoost trains decision trees with a depth of one, that is, trees that make a single question, which we call stumps.

In [ ]:
ada = AdaBoostClassifier(
    estimator=None,
    n_estimators=50,
    learning_rate=1.0,
    random_state=0,
)

ada.fit(X_train, y_train)

In [ ]:
ada.estimator_

### The predictions

The model returns the class with `predict`, and the probability of each class with `predict_proba`.

In [ ]:
ada.predict(X_test)[:10]

In [ ]:
ada.predict_proba(X_test)[:5].round(4)

Each tree of the ensemble votes for a class, and the vote of each tree weighs as much as the accuracy that the tree showed during training, which is the value that AdaBoost stores in `estimator_weights_`. To obtain the probabilities, AdaBoost adds up the weights of the trees that voted for each class and then divides by the total weight, so that the two numbers of each row add up to one.

This means that the probability is really a measure of the agreement between the trees, and not an estimate of how often a client like this one subscribes. The probabilities are therefore not calibrated, and in this dataset they never leave the interval between 0.19 and 0.69, even for the clients that the model is most sure about.

In [ ]:
ada.estimator_weights_[:5].round(3)

### Evaluate the model

In [ ]:
def evaluate(model, X_train, y_train, X_test, y_test):
    for name, X, y in [("train", X_train, y_train), ("test", X_test, y_test)]:
        balanced_accuracy = balanced_accuracy_score(y, model.predict(X))
        roc_auc = roc_auc_score(y, model.predict_proba(X)[:, 1])
        print(f"{name} balanced accuracy: {balanced_accuracy:.4f}, roc-auc: {roc_auc:.4f}")


evaluate(ada, X_train, y_train, X_test, y_test)

### Performance against the number of trees

AdaBoost adds the trees one after the other, so we can evaluate the ensemble after each new tree with `staged_predict_proba`, and we obtain the whole curve by training the model only once.

In [ ]:
ada = AdaBoostClassifier(n_estimators=300, random_state=0)
ada.fit(X_train, y_train)

train_curve = [roc_auc_score(y_train, p[:, 1]) for p in ada.staged_predict_proba(X_train)]
test_curve = [roc_auc_score(y_test, p[:, 1]) for p in ada.staged_predict_proba(X_test)]

len(train_curve)

In [ ]:
trees = range(1, len(train_curve) + 1)

plt.plot(trees, train_curve, label="train")
plt.plot(trees, test_curve, label="test")
plt.xlabel("Number of trees")
plt.ylabel("roc-auc")
plt.title("Performance against the number of trees")
plt.legend()
plt.show()

The performance improves very quickly with the first trees and then it continues to improve slowly, and the two curves stay close to each other, which tells us that the ensemble of stumps is not overfitting. This is different from what we saw with the random forest, where the train curve reached a perfect score almost immediately.

### The learning rate and the number of trees

The learning rate multiplies the contribution of each tree, so with a small learning rate every tree moves the ensemble a little, and we need more trees to arrive at the same place. We train the model with different learning rates and we follow the performance in the test set tree after tree.

In [ ]:
learning_rates = [0.05, 0.1, 0.5, 1.0]

curves = {}
for learning_rate in learning_rates:
    model = AdaBoostClassifier(n_estimators=500, learning_rate=learning_rate, random_state=0)
    model.fit(X_train, y_train)
    curves[learning_rate] = [roc_auc_score(y_test, p[:, 1]) for p in model.staged_predict_proba(X_test)]

curves = pd.DataFrame(curves, index=range(1, 501))
curves.tail(1).round(4)

In [ ]:
curves.plot()
plt.xlabel("Number of trees")
plt.ylabel("roc-auc in the test set")
plt.title("Performance for different learning rates")
plt.legend(title="learning rate")
plt.show()

The curves of the small learning rates climb slowly and they are still climbing when we run out of trees, while the curves of the large learning rates arrive at a good performance with few trees. The two hyperparameters compensate each other, so when we lower the learning rate we need to raise the number of trees, and this is why we always tune them together.

A small learning rate with enough trees usually gives us a slightly better and more stable model, and the price that we pay is the training time.